# Weather API Data

This notebook follows the process of gathering and transforming collected weather data for each destination. Data was gathered from Open Meteo's Historical Weather API. Particular focus will be placed on data transformation, converting the data into a clean, unified, and standardised format. 

## Import Python Libraries

In [2]:
#data handling 
import pandas as pd 
import numpy as np 

#file and path management 
from pathlib import Path as path 

## Load and Concat Each Dataset

In [25]:
#define path to raw nightlife data
weather_raw = path("data")/"raw_data"/"weather" #create a path object 

#create a list of destinations to loop through 
destinations = ['ibiza','mykonos','tenerife','bali','bangkok','cancun',
                'dubrovnik','holetown','maldives','miami','phuket','sal']

#empty list for files to be added to 
destination_files = []

#for loop to locate each individual file and concat to a df 
for d in destinations: 
    
    #list of files to loop through 
    csv_file = weather_raw/f"{d}_weather_data.csv"
    #convert each file to a df
    df = pd.read_csv(csv_file) 
    df.columns = df.iloc[1] #make row 1 the header 
    df = df.iloc[2:].reset_index(drop=True) #data from row 2 onwards
    df['destination'] = d.title() #add column with destination name to join with other destination dfs  
    destination_files.append(df) #add each df to files list 

weather_data = pd.concat(destination_files, ignore_index=True) 
print(f"Total rows loaded: {len(weather_data)}")

Total rows loaded: 1464


## EDA

In [ ]:
weather_data.info()

In [ ]:
weather_data.isna().sum()

## Transform Data

1. Drop unwanted columns
2. Convert columns to int
3. Convert `'sunshine_duration (s)'` to hours
4. Create a month column
5. Rename columns
6. Aggregate and export weather data

### 1. Drop unwanted columns

In [26]:
#drop unwanted cols 
weather_data = weather_data[['time','temperature_2m_max (°C)','sunshine_duration (s)','precipitation_hours (h)','destination']]

### 2. Convert numerical cols to integers

In [27]:
#list on numerical columns 
num_cols = ['temperature_2m_max (°C)','sunshine_duration (s)','precipitation_hours (h)']

#loop through columns and convert
for col in num_cols: 
    weather_data[col] = (
       weather_data[col].astype(float) 
    )

### 3. Convert `'sunshine_duration (s)'` to hours

In [28]:
weather_data['sunshine_duration (s)'] = (weather_data['sunshine_duration (s)'] / 3600).round(1)

### 4. Create a month column

In [29]:
#extract month from time column 
weather_data['month'] = pd.to_datetime(weather_data['time'], format='%Y-%m-%d').dt.month

#replace number value with abv month name 
month_abv = {
    9: 'Sep',
    10: 'Oct',
    11: 'Nov',
    12: 'Dec'
}
weather_data["month"] = weather_data["month"].replace(month_abv)

#drop time column 

weather_data = weather_data.drop(columns='time')

### 5. Rename columns

In [30]:
#rename df columns 
weather_data = weather_data.rename(columns={
    'temperature_2m_max (°C)': 'maxTemperature', 
    'sunshine_duration (s)': 'sunshineHours',
    'precipitation_hours (h)': 'precipitationHours'
})

### 6. Aggregate and export weather data

#### 6a. Aggregate and export weather data (destination x month)

This dataframe will be used for charts in the app, where users can view weather metrics by month of travel.

In [34]:
#group by cols 
group_by_cols = ['destination','month']

#aggregation dictionary assigning calculation to column 
agg_dict = {
    'maxTemperature': ('maxTemperature','mean'), 
    'sunshineHours': ('sunshineHours','mean'), 
    'precipitationHours': ('precipitationHours','mean')
}

weather_data_agg_chart = weather_data.groupby(group_by_cols).agg(
    **agg_dict).reset_index()

#round numerical cols 

num_cols = ['maxTemperature','sunshineHours','precipitationHours']

for col in num_cols: 
    weather_data_agg_chart[col] = weather_data_agg_chart[col].round(1)

In [36]:
#export file 
weather_data_agg_chart.to_csv('destinations_weather_agg_chart', index=False)

#### 6b. Aggregate and export weather data (destination)

This dataframe will be joined with other dataframes (nightlife and flight) on `'destination'`, maintaining a consistent grain with the other dataframes.

In [38]:
#aggregation dictionary assigning calculation to column 
agg_dict = {
    'maxTemperature': ('maxTemperature','mean'), 
    'sunshineHours': ('sunshineHours','mean'), 
    'precipitationHours': ('precipitationHours','mean')
}

destinations_weather_agg = weather_data.groupby('destination').agg(
    **agg_dict).reset_index()

#round numerical cols 

num_cols = ['maxTemperature','sunshineHours','precipitationHours']

for col in num_cols: 
    destinations_weather_agg[col] = destinations_weather_agg[col].round(1)

In [40]:
destinations_weather_agg.to_csv('destinations_weather_agg.csv', index=False)